# Collects samples, find groups of similar attributes, creates directions and modifies a sample in those embeddings.
### Changeable parameters are currently under CONSTANTS (search for CONSTANTS)


In [1]:
from huggingface_hub import login
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import torch

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

CURRENT_FILE_PATH = Path(__file__).resolve() if "__file__" in locals() else Path.cwd()
PROJECT_ROOT = CURRENT_FILE_PATH.parent if CURRENT_FILE_PATH.name == "notebooks" else CURRENT_FILE_PATH

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DIRECTIONS_DIR = PROJECT_ROOT / "notebooks" / "directions"
RENDERS_DIR = PROJECT_ROOT / "notebooks" / "renders"

In [2]:
from datasets import load_dataset
from scr.dataset.utilities import iter_local_subset
from scr.dataset.utilities import download_emilia_subset
from scr.dataset.utilities import get_sample_text, extract_audio_np_and_sr

# CONSTANTS
DOWNLOAD_DATASET = False
LOAD_LOCAL_DATASET = True
DATASET_PATH = "emilia_subset_seed123_n1000"

# Only English samples
if DOWNLOAD_DATASET:
    subset_dir = download_emilia_subset(
        n=1000,
        seed=123,
        out_dir=DATASET_PATH,
    )

if LOAD_LOCAL_DATASET:
    ds = iter_local_subset(DATASET_PATH)
else:
    # Stream one English sample — no full download
    ds = load_dataset(
        "amphion/Emilia-Dataset",
        split="train",
        streaming=True,
    )



In [3]:
sample = next(iter(ds))

print("Keys:", list(sample.keys()))
print("Text:", get_sample_text(sample))

wav, sr = extract_audio_np_and_sr(sample)
print(f"Sample rate: {sr} Hz, length: {len(wav)} samples")

# transform audio to tensor and add dimension
audio_tensor = torch.tensor(wav).unsqueeze(0).float()
print(f"Audio tensor: {audio_tensor.shape}, sr={sr}")

Keys: ['i', 'key', 'text', 'sr', 'num_samples', 'path', 'url', 'seed', 'wav']
Text:  Before we get onto that, there is another noticeable change you'll have probably spotted.
Sample rate: 24000 Hz, length: 101280 samples
Audio tensor: torch.Size([1, 101280]), sr=24000


In [4]:
from TTS.api import TTS

# bypass coqui aggrement
os.environ["COQUI_TOS_AGREED"] = "1"
# Downloads and caches to ~/.local/share/tts/ on first run
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
model = tts.synthesizer.tts_model
print("Model loaded:", type(model).__name__)

Model loaded: Xtts


In [5]:
gpt_cond_latent = model.get_gpt_cond_latents(audio_tensor, sr, length=model.config.gpt_cond_len)
speaker_embedding = model.get_speaker_embedding(audio_tensor, sr)

print("gpt_cond_latent shape:", gpt_cond_latent.shape)
print("speaker_embedding shape:", speaker_embedding.shape)

gpt_cond_latent shape: torch.Size([1, 32, 1024])
speaker_embedding shape: torch.Size([1, 512, 1])


---

# Functions for the different desired attributes (Pitch, Loudness, Rate of speech)

In [6]:
import librosa

def compute_pitch_simple(sample):
    wav, sr = extract_audio_np_and_sr(sample)
    f0 = librosa.yin(
            wav,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr
        )
    mean_pitch = float(np.mean(f0[f0 > 0])) if np.any(f0 > 0) else 0.0
    return mean_pitch

In [7]:
def compute_loudness_simple(sample):
    wav, sr = extract_audio_np_and_sr(sample)

    rms = np.sqrt(np.mean(wav**2))

    # Convert to Decibels (dBFS)
    if rms > 1e-10:
        return 20 * np.log10(rms)
    else:
        return -100.0  # Represents silence


import pyloudnorm as pyln

def compute_loudness_advanced(sample):
    # Perceived loudness
    wav, sr = extract_audio_np_and_sr(sample)

    meter = pyln.Meter(sr)

    # Values are usually negative (e.g., -23.0 is common)
    loudness = meter.integrated_loudness(np.asarray(wav, dtype=np.float32))

    return loudness

In [8]:
import re

def compute_speech_rate_simple(sample):
    # cps: Characters per second
    # Using this because non-English might not have spaces (Chinese, Japanese)

    wav, sr = extract_audio_np_and_sr(sample)
    text = get_sample_text(sample)

    # Clean text: Remove punctuation and extra whitespace
    clean_text = re.sub(r'[^\w\s]', '', text).replace(" ", "")
    duration = len(wav) / sr if sr > 0 else 0.0

    # (Characters per second) CPS Calculation
    cps = (len(clean_text) / duration) if duration > 0 else 0.0
    return cps

# Recommended Logic
# TODO work in progress
# https://gemini.google.com/app/aefb2b36771c80d3
def compute_speech_rate_advanced(sample, vad_model, phonemizer_obj):
    wav, sr = extract_audio_np_and_sr(sample)
    text = get_sample_text(sample)

    # Get Active Duration (Removes silence)
    speech_timestamps = vad_model.get_speech_timestamps(wav, sr)
    active_duration = sum([t['end'] - t['start'] for t in speech_timestamps]) / sr

    if active_duration <= 0: return 0.0

    # Convert to Phonemes
    phonemes = phonemizer_obj.phonemize([text], strip=True)[0]
    phoneme_count = len(phonemes.replace(" ", ""))

    # (Phonemes per second) PPS Calculation
    pps = phoneme_count / active_duration
    return pps


---

# Temporary Example extracting "directions" for pitch etc.

In [9]:
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional
import shutil

def load_or_generate_directions(
        *,
        create_new: bool,
        ds: Optional[Iterable[dict]] = None,
        model: Optional[Any] = None,
        max_scan: int = 1000,
        pct: float = 0.15,
        save_dir: str | Path = "directions",
        load_path: str | Path = None,
        save: bool = True,
        store_ordered_samples: bool = True,
        filename_prefix: str = "xtts_directions",
        device: Optional[str] = None,
) -> Dict[str, Any]:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if not create_new:
        load_path = Path(load_path)
        payload = torch.load(load_path, map_location=device)
        print(f"Loaded directions from: {load_path}")
        return payload

    # --- Collect representations + stats ---
    all_latents: list[torch.Tensor] = []
    all_spk_embs: list[torch.Tensor] = []
    stats: list[dict] = []
    audio_src_paths: list[Optional[str]] = []

    print(f"Scanning {max_scan} samples... (pct={pct:.3f}, device={device})")

    processed = 0
    for i, sample in enumerate(ds):
        if i >= max_scan:
            break

        audio_array, sr = extract_audio_np_and_sr(sample)

        pitch = compute_pitch_simple(sample)
        loudness = compute_loudness_simple(sample)
        cps = compute_speech_rate_simple(sample)

        audio_tensor = torch.tensor(audio_array).unsqueeze(0).float().to(device)
        with torch.no_grad():
            latent = model.get_gpt_cond_latents(audio_tensor, sr, length=model.config.gpt_cond_len)
            all_latents.append(latent)
            spk = model.get_speaker_embedding(audio_tensor, sr)
            all_spk_embs.append(spk)

        stats.append({"cps": float(cps), "pitch": float(pitch), "loudness": float(loudness)})
        audio_src_paths.append(sample.get("path", None))
        processed += 1

        if (processed % 25 == 0):
            print(f"  Processed {processed}/{max_scan}...")

    def _direction_for(feature_name: str, reps: list[torch.Tensor]) -> torch.Tensor:
        values = np.array([s[feature_name] for s in stats], dtype=np.float64)
        indices = np.argsort(values)

        num = max(1, int(processed * pct))
        low_idx = indices[:num]
        high_idx = indices[-num:]

        print(f"Calculating {feature_name}: averaging {len(low_idx)} low vs {len(high_idx)} high samples")

        mean_low = torch.stack([reps[int(j)] for j in low_idx]).mean(dim=0)
        mean_high = torch.stack([reps[int(j)] for j in high_idx]).mean(dim=0)
        return mean_high - mean_low

    def _copy_ordered_samples(*, feature: str, out_dir: Path) -> None:
        out_dir.mkdir(parents=True, exist_ok=True)

        values = np.array([s[feature] for s in stats], dtype=np.float64)
        indices = np.argsort(values)  # ascending

        copied = 0
        skipped = 0

        for rank, j in enumerate(indices):
            src_rel = audio_src_paths[int(j)]
            if not src_rel:
                skipped += 1
                print(f"Sample {j} has no audio path; skipping")
                continue

            src_path = Path(src_rel)
            if not src_path.is_absolute():
                src_path = Path.cwd() / src_path

            if not src_path.exists():
                skipped += 1
                print(f"Sample {j} audio path does not exist: {src_path}; skipping")
                continue

            key = str(sample.get("key", "sample")) if isinstance(sample, dict) else "sample"
            val = float(values[int(j)])
            dst_name = f"{rank:04d}_{src_path.stem}_{feature}={val:.4f}{src_path.suffix}"
            dst_path = out_dir / dst_name

            if not dst_path.exists():
                shutil.copy2(src_path, dst_path)
                copied += 1

        print(f"Ordered samples saved for '{feature}': copied={copied}, skipped={skipped}, dir={out_dir}")


    # GPT-latent directions (existing behavior)
    speed_direction = _direction_for("cps", all_latents)
    pitch_direction = _direction_for("pitch", all_latents)
    loudness_direction = _direction_for("loudness", all_latents)
    spk_speed_direction = _direction_for("cps", all_spk_embs)
    spk_pitch_direction = _direction_for("pitch", all_spk_embs)
    spk_loudness_direction = _direction_for("loudness", all_spk_embs)

    payload: Dict[str, Any] = {
        "stats": stats,
        "speed_direction": speed_direction.detach(),
        "pitch_direction": pitch_direction.detach(),
        "loudness_direction": loudness_direction.detach(),
        "all_latents": [t.detach().cpu() for t in all_latents],
        "spk_speed_direction": spk_speed_direction.detach(),
        "spk_pitch_direction": spk_pitch_direction.detach(),
        "spk_loudness_direction": spk_loudness_direction.detach(),
        "all_spk_embs": [t.detach().cpu() for t in all_spk_embs],
    }

    created_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    pct_label = f"{pct * 100:.1f}".replace(".", "p")
    filename = f"{filename_prefix}_N{processed}_pct{pct_label}_{created_at}.pt"

    if store_ordered_samples:
        ordered_root = PROJECT_ROOT / "notebooks" / DATASET_PATH / "ordered_samples" / f"{filename_prefix}_N{processed}_{created_at}"
        _copy_ordered_samples(feature="pitch", out_dir=ordered_root / "pitch")
        _copy_ordered_samples(feature="loudness", out_dir=ordered_root / "loudness")
        _copy_ordered_samples(feature="cps", out_dir=ordered_root / "speech_rate")

    if save:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / filename
        torch.save(payload, save_path)
        print(f"Saved directions to: {save_path}")

    return payload


In [10]:
# CONSTANTS
CREATE_NEW = False
SAVE = False
STORE_ORDERED_SAMPLES = False

# TODO this is very manual still
target_file = "xtts_directions_N999_pct15p0_20260302_203148.pt"
full_load_path = DIRECTIONS_DIR / target_file

directions = load_or_generate_directions(
    create_new=CREATE_NEW,
    save=SAVE,
    store_ordered_samples = STORE_ORDERED_SAMPLES,
    ds=ds,
    model=model,
    max_scan=1000,
    pct=0.15,
    save_dir=DIRECTIONS_DIR,
    load_path=full_load_path,
)

speed_direction = directions["speed_direction"]
pitch_direction = directions["pitch_direction"]
loudness_direction = directions["loudness_direction"]
spk_speed_direction = directions.get("spk_speed_direction", None)
spk_pitch_direction = directions.get("spk_pitch_direction", None)
spk_loudness_direction = directions.get("spk_loudness_direction", None)

Loaded directions from: C:\Users\Andre\PycharmProjects\Speech-technology-project-7\notebooks\directions\xtts_directions_N999_pct15p0_20260302_203148.pt


In [11]:
print(torch.norm(speed_direction).item())
print(torch.norm(pitch_direction).item())
print(torch.norm(spk_speed_direction).item())
print(torch.norm(spk_pitch_direction).item())

34.3885498046875
94.22151947021484
0.29746153950691223
0.6107646226882935


In [12]:
from dataclasses import dataclass

@dataclass
class DirectionsBundle:
    # GPT (Latent) Directions/Targets
    speed_direction: torch.Tensor | None = None
    pitch_direction: torch.Tensor | None = None
    loudness_direction: torch.Tensor | None = None
    speed_low: torch.Tensor | None = None
    speed_high: torch.Tensor | None = None
    pitch_low: torch.Tensor | None = None
    pitch_high: torch.Tensor | None = None
    loudness_low: torch.Tensor | None = None
    loudness_high: torch.Tensor | None = None

    # Speaker Embedding Directions/Targets
    spk_speed_direction: torch.Tensor | None = None
    spk_pitch_direction: torch.Tensor | None = None
    spk_loudness_direction: torch.Tensor | None = None
    spk_speed_low: torch.Tensor | None = None
    spk_speed_high: torch.Tensor | None = None
    spk_pitch_low: torch.Tensor | None = None
    spk_pitch_high: torch.Tensor | None = None
    spk_loudness_low: torch.Tensor | None = None
    spk_loudness_high: torch.Tensor | None = None


def slerp(val, low, high):
    """
    Spherical linear interpolation between two tensors (low and high)
    based on 'val' (0.0 to 1.0).
    """
    low_norm = low / torch.norm(low)
    high_norm = high / torch.norm(high)

    dot = torch.sum(low_norm * high_norm)
    dot = torch.clamp(dot, -1.0, 1.0)
    omega = torch.acos(dot)
    so = torch.sin(omega)

    if so < 1e-6:
        return (1.0 - val) * low + val * high

    return (torch.sin((1.0 - val) * omega) / so) * low + (torch.sin(val * omega) / so) * high


def get_anchor_points(feature_name: str, reps: list[torch.Tensor], stats_list: list[dict], *, pct: float = 0.15):
    values = [s[feature_name] for s in stats_list]
    indices = np.argsort(values)
    num = max(1, int(len(indices) * pct))

    low_idx = indices[:num]
    high_idx = indices[-num:]

    print(f"Anchoring {feature_name}: using {len(low_idx)} samples for Low and High points")

    mean_low = torch.stack([reps[int(i)] for i in low_idx]).mean(dim=0)
    mean_high = torch.stack([reps[int(i)] for i in high_idx]).mean(dim=0)
    return mean_low, mean_high


stats = directions.get("stats", None)
all_latents = directions.get("all_latents", None)
all_spk_embs = directions.get("all_spk_embs", None)

speed_low, speed_high = get_anchor_points("cps", all_latents, stats, pct=0.15)
pitch_low, pitch_high = get_anchor_points("pitch", all_latents, stats, pct=0.15)
loudness_low, loudness_high = get_anchor_points("loudness", all_latents, stats, pct=0.15)
spk_speed_low, spk_speed_high = get_anchor_points("cps", all_spk_embs, stats, pct=0.15)
spk_pitch_low, spk_pitch_high = get_anchor_points("pitch", all_spk_embs, stats, pct=0.15)
spk_loudness_low, spk_loudness_high = get_anchor_points("loudness", all_spk_embs, stats, pct=0.15)

directions_bundle = DirectionsBundle(
    speed_direction=speed_direction,
    pitch_direction=pitch_direction,
    loudness_direction=loudness_direction,
    speed_low=speed_low,
    speed_high=speed_high,
    pitch_low=pitch_low,
    pitch_high=pitch_high,
    loudness_low=loudness_low,
    loudness_high=loudness_high,
    spk_speed_direction=spk_speed_direction,
    spk_pitch_direction=spk_pitch_direction,
    spk_loudness_direction=spk_loudness_direction,
    spk_speed_low=spk_speed_low,
    spk_speed_high=spk_speed_high,
    spk_pitch_low=spk_pitch_low,
    spk_pitch_high=spk_pitch_high,
    spk_loudness_low=spk_loudness_low,
    spk_loudness_high=spk_loudness_high,
)


Anchoring cps: using 149 samples for Low and High points
Anchoring pitch: using 149 samples for Low and High points
Anchoring loudness: using 149 samples for Low and High points
Anchoring cps: using 149 samples for Low and High points
Anchoring pitch: using 149 samples for Low and High points
Anchoring loudness: using 149 samples for Low and High points


In [13]:
from IPython.display import Audio

def compute_modified_embeddings(
    *,
    base_gpt_cond_latent: torch.Tensor,
    base_speaker_embedding: torch.Tensor,
    directions_bundle: DirectionsBundle,
    gpt_method: str | None,
    spk_method: str | None,
    speed: float,
    pitch: float,
    loudness: float,
    normalize_speaker_embedding: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:

    s = float(speed)
    p = float(pitch)
    l = float(loudness)
    latent = base_gpt_cond_latent.clone()
    spk = base_speaker_embedding.clone()

    if gpt_method == "linear":
        latent = base_gpt_cond_latent + (s * directions_bundle.speed_direction) + (p * directions_bundle.pitch_direction) + (l * directions_bundle.loudness_direction)
    elif gpt_method == "spherical":
        if s != 0:
                target = directions_bundle.speed_high if s > 0 else directions_bundle.speed_low
                latent = slerp(min(abs(s), 1.0), latent, target)
        if p != 0:
            target = directions_bundle.pitch_high if p > 0 else directions_bundle.pitch_low
            latent = slerp(min(abs(p), 1.0), latent, target)
        if l != 0:
            target = directions_bundle.loudness_high if l > 0 else directions_bundle.loudness_low
            latent = slerp(min(abs(l), 1.0), latent, target)
    if spk_method == "linear":
        spk = base_speaker_embedding + (s * directions_bundle.spk_speed_direction) + (p * directions_bundle.spk_pitch_direction) + (l * directions_bundle.spk_loudness_direction)
        if normalize_speaker_embedding:
            spk = torch.nn.functional.normalize(spk, p=2, dim=1)
    elif spk_method == "spherical":
        if s != 0:
            target = directions_bundle.spk_speed_high if s > 0 else directions_bundle.spk_speed_low
            spk = slerp(min(abs(s), 1.0), spk, target)
        if p != 0:
            target = directions_bundle.spk_pitch_high if p > 0 else directions_bundle.spk_pitch_low
            spk = slerp(min(abs(p), 1.0), spk, target)
        if l != 0:
            target = directions_bundle.spk_loudness_high if l > 0 else directions_bundle.spk_loudness_low
            spk = slerp(min(abs(l), 1.0), spk, target)
        if normalize_speaker_embedding:
            spk = torch.nn.functional.normalize(spk, p=2, dim=1)

    return latent, spk


def synthesize(
    *,
    model: Any,
    base_gpt_cond_latent: torch.Tensor,
    base_speaker_embedding: torch.Tensor,
    directions_bundle: DirectionsBundle,
    gpt_method: str | None,
    spk_method: str | None,
    speed: float,
    pitch: float,
    loudness: float,
) -> dict:

    modified_latent, modified_spk = compute_modified_embeddings(
        base_gpt_cond_latent=base_gpt_cond_latent,
        base_speaker_embedding=base_speaker_embedding,
        directions_bundle=directions_bundle,
        gpt_method=gpt_method,
        spk_method=spk_method,
        speed=speed,
        pitch=pitch,
        loudness=loudness,
    )

    with torch.no_grad():
        out = model.inference(
            #text="Hello! This is how I sound after you modified my embeddings.",
            text="The birch canoe slid on the smooth planks, but the sudden wind blew it off course.",
            language="en",
            gpt_cond_latent=modified_latent,
            speaker_embedding=modified_spk,
            temperature=model.config.temperature,
            length_penalty=model.config.length_penalty,
            repetition_penalty=model.config.repetition_penalty,
            top_k=model.config.top_k,
            top_p=model.config.top_p,
        )

    return out

In [14]:
# UI Elements
from ipywidgets import widgets
from IPython.display import Audio, display, clear_output
s_slider = widgets.FloatSlider(value=0, min=-10, max=10, step=0.2, description="Speed 🏃")
p_slider = widgets.FloatSlider(value=0, min=-10, max=10, step=0.2, description="Pitch 🎤")
l_slider = widgets.FloatSlider(value=0, min=-10, max=10, step=0.2, description="Loudness 🔊")
apply_to = widgets.ToggleButtons(options=[("Conditional Latent", "gpt"), ("Speaker Embedding", "spk")], value="gpt", description="Apply to:",)
ui_out = widgets.Output()

gpt_mode_sel = widgets.Dropdown(
    options=[("None", None), ("Linear", "linear"), ("Spherical", "spherical")],
    value="linear",
    description="GPT Method:"
)
spk_mode_sel = widgets.Dropdown(
    options=[("None", None), ("Linear", "linear"), ("Spherical", "spherical")],
    value=None,
    description="SPK Method:"
)
gen_btn = widgets.Button(description="Generate Audio", button_style="success", icon="play")

def on_click_generate(_b):
    with ui_out:
        clear_output()
        print(f"Generating: GPT={gpt_mode_sel.value}, SPK={spk_mode_sel.value}...")
        out = synthesize(
            model=model,
            base_gpt_cond_latent=gpt_cond_latent,
            base_speaker_embedding=speaker_embedding,
            directions_bundle=directions_bundle,
            gpt_method=gpt_mode_sel.value,
            spk_method=spk_mode_sel.value,
            speed=s_slider.value,
            pitch=p_slider.value,
            loudness=l_slider.value,
        )
        display(Audio(out["wav"], rate=24000, autoplay=True))

gen_btn.on_click(on_click_generate)

display(widgets.VBox([
    widgets.HBox([gpt_mode_sel, spk_mode_sel]),
    s_slider, p_slider, l_slider,
    gen_btn, ui_out
]))

In [15]:
from pathlib import Path
from typing import Any, Literal, Sequence
from scipy.io import wavfile

Control = Literal["speed", "pitch", "loudness", "none"]
Method = Literal["linear", "spherical"]

def create_many_samples(
    *,
    model: Any,
    base_gpt_cond_latent: torch.Tensor,
    base_speaker_embedding: torch.Tensor,
    directions_bundle=directions_bundle,
    gpt_method: str | None,
    spk_method: str | None,
    n_per_setting: int,
    values: Sequence[float],
    control: Control,
    seed: int | None = None,           # set for reproducibility
    save_dir: str | Path | None = None,
):
    """
    Generate many audio samples for a sweep of values.

    Inputs
      - n_per_setting: how many samples per value
      - values: list of values to use, e.g. [-1, -0.5, 0, 0.5, 2]
      - control: "speed" or "pitch" or "none"
      - method: "linear" or "spherical"
    """

    if control == "none":
        sweep_values = [0.0]
    else:
        sweep_values = [float(v) for v in values]

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

    base_seed = seed
    for rep_idx in range(n_per_setting):
        for setting_idx, v in enumerate(sweep_values):
            if base_seed is not None:
                cur_seed = int(base_seed + setting_idx * 10_000 + rep_idx)
                torch.manual_seed(cur_seed)
                np.random.seed(cur_seed % (2**32 - 1))

            speed = 0.0
            pitch = 0.0
            loudness = 0.0
            if control == "speed":
                speed = float(v)
            elif control == "pitch":
                pitch = float(v)
            elif control == "loudness":
                loudness = float(v)

            if save_dir is not None:
                fname = (
                    f"gpt-{gpt_method}_"
                    f"spk-{spk_method}_"
                    f"{control}_"
                    f"{float(v):+.3f}_"
                    f"rep{rep_idx:02d}.wav"
                )
                path = save_dir / fname

                # Only synthesize and save a sample if it doesn't exist yet
                if not path.exists():
                    out = synthesize(
                        model=model,
                        base_gpt_cond_latent=base_gpt_cond_latent,
                        base_speaker_embedding=base_speaker_embedding,
                        directions_bundle=directions_bundle,
                        gpt_method=gpt_method,
                        spk_method=spk_method,
                        speed=speed,
                        pitch=pitch,
                        loudness=loudness,
                    )
                    wav = np.asarray(out["wav"], dtype=np.float32)
                    wavfile.write(path, 24000, wav)

    return

In [16]:
'''
Valid values
------------------------------------------
Linear:

speed
Plus Seems to speed up recording
+5 sounds like male
+10 sounds like male
+100 too much, garbage
-100 too much, garbage

pitch
Seems to hear most difference in range 2, 3, 5, 8
More is not helpful
+50 too much, garbage
-50 too much, garbage
-------------------------------------

Spehrical:
Valid values are -1 to 1. At that point they reached their targets.

'''

'\nValid values\n------------------------------------------\nLinear:\n\nspeed\nPlus Seems to speed up recording\n+5 sounds like male\n+10 sounds like male\n+100 too much, garbage\n-100 too much, garbage\n\npitch\nSeems to hear most difference in range 2, 3, 5, 8\nMore is not helpful\n+50 too much, garbage\n-50 too much, garbage\n-------------------------------------\n\nSpehrical:\nValid values are -1 to 1. At that point they reached their targets.\n\n'

In [63]:
create_many_samples(
    model=model,
    base_gpt_cond_latent=gpt_cond_latent,
    base_speaker_embedding=speaker_embedding,
    directions_bundle=directions_bundle,
    gpt_method="none",                     # "linear", "spherical" or none
    spk_method="spherical",                  # "linear", "spherical" or none
    n_per_setting=1,
    values=[-1, -0.75, -0.50, -0.30, -0.20, -0.10, -0.05, -0.03, -0.02, -0.01, 0, 0.01, 0.02, 0.03, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1],
    #values=[-3, -2, -1, -0.5, 0, 0.5, 1, 2, 3],
    #values=[-100, -75, -50, -25, -10, -8, -5, -4, -3, -2.5, -2, -1.5, -1, -0.75, -0.5, -0.2, -0.1, -0.05, -0.01, 0, 0.01, 0.05, 0.1, 0.2, 0.5, 0.75, 1, 1.5, 2, 2.5, 3, 4, 5, 8, 10, 50, 75, 100],
    control="loudness",                     # "speed", "pitch" or loudness
    seed=123,
    save_dir=RENDERS_DIR,
)


In [18]:
from scipy.io import wavfile

def save_anchor_samples(
    model: Any,
    bundle: DirectionsBundle,
    feature: Literal["speed", "pitch", "loudness"],
):
    save_path = Path("anchor_samples")
    save_path.mkdir(parents=True, exist_ok=True)

    # Mapping features to bundle attributes
    mapping = {
        "speed": (bundle.speed_low, bundle.speed_high, bundle.spk_speed_low, bundle.spk_speed_high),
        "pitch": (bundle.pitch_low, bundle.pitch_high, bundle.spk_pitch_low, bundle.spk_pitch_high),
        "loudness": (bundle.loudness_low, bundle.loudness_high, bundle.spk_loudness_low, bundle.spk_loudness_high),
    }

    g_low, g_high, s_low, s_high = mapping[feature]
    anchors = [("low", g_low, s_low), ("high", g_high, s_high)]

    for label, g_latent, s_emb in anchors:
        print(f"Generating {feature} {label} anchor...")

        with torch.no_grad():
            out = model.inference(
                text="The birch canoe slid on the smooth planks, but the sudden wind blew it off course.",
                language="en",
                gpt_cond_latent=g_latent.clone(),
                speaker_embedding=s_emb.clone(),
                temperature=model.config.temperature,
                length_penalty=model.config.length_penalty,
                repetition_penalty=model.config.repetition_penalty,
                top_k=model.config.top_k,
                top_p=model.config.top_p,
            )

        # Save to disk
        filename = f"anchor_{feature}_{label}.wav"
        file_full_path = save_path / filename
        wav_data = np.asarray(out["wav"], dtype=np.float32)
        wavfile.write(file_full_path, 24000, wav_data)
        print(f"Saved: {file_full_path}")

# Execute for all three features
#for feat in ["speed", "pitch", "loudness"]:
#    save_anchor_samples(model, directions_bundle, feat)

In [52]:
%matplotlib tk

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def validate_modifications(renders_dir, control_type, gpt_method, spk_method, min, max):
    """
    Scans the renders directory, measures the audio, and plots Intended vs. Measured.
    """
    results = []
    render_path = Path(renders_dir)

    # Filter files for the specific control and method we want to validate
    # Example filename: gpt-linear_spk-none_pitch_+2.500_rep00.wav
    files = list(render_path.glob(f"*{control_type}*.wav"))

    if not files:
        print(f"No files found for control: {control_type}")
        return

    print(f"Analyzing {len(files)} samples...")

    for f in files:
        if f"gpt-{gpt_method}" not in f.name: continue
        if f"spk-{spk_method}" not in f.name: continue
        parts = f.stem.split('_')
        try:
            # Finding the index of the control tag and taking the next element
            idx = parts.index(control_type)
            intended_val = float(parts[idx + 1])
            if not (min <= intended_val <= max): continue
        except (ValueError, IndexError):
            continue

        wav, sr = librosa.load(f, sr=None)
        mock_sample = {
            "mp3": {
                "array": wav,
                "sampling_rate": sr
            },
            "text": "The birch canoe slid on the smooth planks, but the sudden wind blew it off course."
        }
        # Measure
        if control_type == "pitch":
            measured = compute_pitch_simple(mock_sample)
        elif control_type == "loudness":
            measured = compute_loudness_simple(mock_sample)
        elif control_type == "speed":
            measured = compute_speech_rate_simple(mock_sample)
        else:
            measured = 0

        results.append({
            "Intended Shift": intended_val,
            "Measured Value": measured,
            "File": f.name
        })

    df = pd.DataFrame(results).sort_values("Intended Shift")

    # --- Plotting ---
    plt.figure(figsize=(10, 6))
    sns.regplot(data=df, x="Intended Shift", y="Measured Value", fit_reg=True, line_kws={"color": "red", "alpha": 0.5})

    baseline_df = df[df["Intended Shift"] == 0]
    plt.scatter(baseline_df["Intended Shift"], baseline_df["Measured Value"],
                    color="black", s=50, marker="o", edgecolor="black",
                    label="Not modified", zorder=5)
    plt.legend()


    plt.title(f"Validation: {control_type.capitalize()} | GPT: {gpt_method} | SPK: {spk_method} | Range: [{min}, {max}]", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    return df

# Run validation for pitch
df_pitch = validate_modifications(RENDERS_DIR,
                                  control_type="pitch",
                                  gpt_method="linear",
                                  spk_method="none",
                                  min=-3,
                                  max=3,
                                )

Analyzing 53 samples...


In [88]:
def plot_validation_matrix_flipped(renders_dir, gpt_method, spk_method, val_min, val_max):
    render_path = Path(renders_dir)
    controls = ["pitch", "loudness", "speed"]
    measured_cols = ["Measured_Pitch", "Measured_Loudness", "Measured_Speed"]
    all_data = []

    row_limits = {}

    row_limits = {
        "Measured_Pitch": (100, 450),    # Example: Hz
        "Measured_Loudness": (-22, -13),  # Example: dB
        "Measured_Speed": (8, 19)        # Example: syllables/sec
    }

    # 1. Collect and Measure
    files = list(render_path.glob("*.wav"))

    for f in files:
        if f"gpt-{gpt_method}" not in f.name or f"spk-{spk_method}" not in f.name:
            continue

        parts = f.stem.split('_')
        current_control = next((c for c in controls if c in parts), None)
        if not current_control:
            continue

        try:
            idx = parts.index(current_control)
            intended_val = float(parts[idx + 1])
        except (ValueError, IndexError):
            continue

        if not (val_min <= intended_val <= val_max):
            continue

        wav, sr = librosa.load(f, sr=None)
        mock_sample = {
            "mp3": {"array": wav, "sampling_rate": sr},
            "text": "The birch canoe slid on the smooth planks, but the sudden wind blew it off course."
        }

        all_data.append({
            "Intended_Control": current_control,
            "Intended_Shift": intended_val,
            "Measured_Speed": compute_speech_rate_simple(mock_sample),
            "Measured_Pitch": compute_pitch_simple(mock_sample),
            "Measured_Loudness": compute_loudness_simple(mock_sample)
        })

    df = pd.DataFrame(all_data)
    if df.empty:
        print("No matching data found.")
        return

    # 2. Setup Plotting Matrix (Rows = Measured, Cols = Intended)
    fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharey='row')
    active_method = f"GPT {gpt_method}" if gpt_method != "none" else f"SPK {spk_method}"
    fig.suptitle(f"Acoustic Control Validation: Target vs. Measured Shifts (Method: {active_method.upper()})", fontsize=16)

    row_labels = ["Pitch (Hz)", "Loudness (dB)", "Speed (syll/sec)"]

    for col_idx, ctrl in enumerate(controls):
        # Isolate the data where we specifically varied 'ctrl'
        subset = df[df["Intended_Control"] == ctrl].sort_values("Intended_Shift")

        for row_idx, meas in enumerate(measured_cols):
            ax = axes[row_idx, col_idx]

            # Check if the current plot is on the diagonal (Pitch-Pitch, Loudness-Loudness, Speed-Speed)
            if row_idx == col_idx:
                ax.set_facecolor('#f0f8ff')  # Light AliceBlue to highlight sensitivity

            if meas in row_limits:
                ax.set_ylim(row_limits[meas])

            if not subset.empty:
                sns.regplot(data=subset, x="Intended_Shift", y=meas, ax=ax,
                            scatter_kws={'alpha':0.4, 'color':'blue'},
                            line_kws={'color':'red', 'linewidth':1})

                # Highlight baseline (Shift = 0)
                baseline = subset[subset["Intended_Shift"] == 0]
                if not baseline.empty:
                    ax.scatter(baseline["Intended_Shift"], baseline[meas], color='black', s=60, zorder=5)

            # Titles and Labels
            if row_idx == 0:
                ax.set_title(f"Varying {ctrl.upper()}", fontweight='bold', pad=15)

            # Label y-axis only on the first column for cleanliness
            if col_idx == 0:
                ax.set_ylabel(row_labels[row_idx], fontweight='bold', fontsize=12)
            else:
                ax.set_ylabel("")

            if row_idx < 2:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel("Intended Shift", fontweight='bold')

            ax.set_xlabel("")
            ax.grid(True, linestyle=':', alpha=0.6)

    fig.supxlabel("Intended Shift Amount", fontweight='bold', fontsize=14, y=0.02)
    fig.supylabel("Measured Value", fontweight='bold', fontsize=14)
    baseline_dot = plt.Line2D([0], [0], marker='o', color='w', label='Unmodified Embedding',markerfacecolor='black', markersize=8)
    fig.legend(handles=[baseline_dot], loc='upper right', bbox_to_anchor=(0.98, 0.96))
    plt.tight_layout(rect=[0.02, 0.03, 1, 0.95])
    plt.show()
# Run the updated validation

plot_validation_matrix_flipped(RENDERS_DIR, gpt_method="linear", spk_method="none", val_min=-3, val_max=3)
plot_validation_matrix_flipped(RENDERS_DIR, gpt_method="none", spk_method="spherical", val_min=-1, val_max=1)

In [22]:
# import matplotlib
# Try 'qt5' if you have PyQt installed, otherwise 'tk' is usually built-in
# %matplotlib tk
#
# import matplotlib.pyplot as plt
#
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from sklearn.decomposition import PCA
# from mpl_toolkits.mplot3d import Axes3D

#
# def visualize_pitch_3d(all_spk_embs, stats, pct=0.15):
#     """
#     Creates a 3D PCA plot of speaker embeddings colored by pitch,
#     with an arrow showing the direction from low to high pitch centroids.
#     """
#     # 1. Convert list of tensors to a single numpy array
#     data = torch.stack([t.detach().cpu().squeeze() for t in all_spk_embs]).numpy()
#     pitches = np.array([s["pitch"] for s in stats])
#
#     # 2. Reduce dimensionality to 3D using PCA
#     pca = PCA(n_components=3)
#     coords = pca.fit_transform(data)
#
#     # --- NEW: Arrow Logic ---
#     # Find indices for low and high pitch
#     indices = np.argsort(pitches)
#     num = max(1, int(len(indices) * pct))
#     low_idx = indices[:num]
#     high_idx = indices[-num:]
#
#     # Calculate centroids in the HIGH-DIMENSIONAL space first
#     mean_low_vec = data[low_idx].mean(axis=0)
#     mean_high_vec = data[high_idx].mean(axis=0)
#
#     # Project these centroids into the 3D PCA space
#     # (Important: use .transform, not .fit_transform)
#     low_centroid_3d = pca.transform(mean_low_vec.reshape(1, -1))[0]
#     high_centroid_3d = pca.transform(mean_high_vec.reshape(1, -1))[0]
#     # ------------------------
#
#     # 3. Plotting
#     fig = plt.figure(figsize=(12, 10))
#     ax = fig.add_subplot(111, projection='3d')
#
#     # --- Formatting: Hide numbers and labels, keep grid ---
#     ax.set_title("Pitch Direction in Speaker Embedding Space", fontsize=15)
#
#     # Remove tick labels (numbers)
#     ax.set_xticklabels([])
#     ax.set_yticklabels([])
#     ax.set_zticklabels([])
#
#     # Remove axis labels (PC 1, PC 2, etc.)
#     ax.set_xlabel("")
#     ax.set_ylabel("")
#     ax.set_zlabel("")
#
#     # Ensure grid remains visible
#     ax.grid(True)
#     # ------------------------------------------------------
#
#     scatter = ax.scatter(
#         coords[:, 0], coords[:, 1], coords[:, 2],
#         c=pitches, cmap='magma', alpha=0.6, edgecolors='none'
#     )
#
#     # --- NEW: Draw the Arrow ---
#     # ax.quiver(x, y, z, dx, dy, dz)
#     ax.quiver(
#         low_centroid_3d[0], low_centroid_3d[1], low_centroid_3d[2],  # Start
#         high_centroid_3d[0] - low_centroid_3d[0],                  # DX
#         high_centroid_3d[1] - low_centroid_3d[1],                  # DY
#         high_centroid_3d[2] - low_centroid_3d[2],                  # DZ
#         color='red', linewidth=4, label='Pitch Direction', arrow_length_ratio=0.1
#     )
#     # ---------------------------
#
#     ax.set_title("3D PCA Projection: Speaker Identity Space with Pitch Vector", fontsize=15)
#     ax.set_xlabel("PC 1")
#     ax.set_ylabel("PC 2")
#     ax.set_zlabel("PC 3")
#
#     cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
#     cbar.set_label('Pitch Value (Measured)', rotation=270, labelpad=15)
#
#     plt.show()
#
# # Usage:
# visualize_pitch_3d(all_spk_embs, stats)